Imports which we will use in the deliverable:

In [35]:
from pathlib import Path
import re

from typing import Any
import json

Task 1: File ingestion and regular expressions.

Part 1 of Task 1: Robust file reading function.

In [19]:
def read_log_file(file_path: Path) -> tuple[str, str]:
    """Read a log file using either Latin-1 or UTF-8 encoding.
    
    Args:
        file_path: Path to the log file.
        
    Returns:
        A tuple containing decoded text and encoding used.
    """

    raw_data = file_path.read_bytes()

    try:
        return raw_data.decode("utf-8"), "utf-8"
    except UnicodeDecodeError:
        return raw_data.decode("latin-1"), "latin-1"

This function reades the bytes and first tries the utf-8 encoding.
If that fails, it will try the Latin-1 encoding.

Part 2 of Task 1: Log parser function.

In [20]:
def parse_log(file_text: str) -> list[dict[str, str | float]]:
    """Extract the transaction records from log text.
    
    Args:
        file_text: Log text including transactions and noise.
        
    Returns:
        A list of dictionaries with timestamp, register_id, item,
        price, currency and status. Prices as floats, other values
        as strings.
    """
    pattern = re.compile(
        r"\[(?P<timestamp>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})\]\s+"
        r"\[(?P<register_id>POS_\d+)\]\s+"
        r"item=(?P<item>.+?)\s+"
        r"price=(?P<price>-?\d+(?:\.\d+)?)\s+"
        r"(?P<currency>[A-Z]{3})\s+"
        r"\((?P<status>SUCCESS|REFUND|CANCELLED)\)"
    )

    transactions: list[dict[str, str | float]] = []

    for line in file_text.splitlines():
        match = pattern.fullmatch(line.strip())

        if match is not None:
            record: dict[str, str | float] = {
                "timestamp": match.group("timestamp"),
                "register_id": match.group("register_id"),
                "item": match.group("item"),
                "price": float(match.group("price")),
                "currency": match.group("currency"),
                "status": match.group("status"),
            }
            transactions.append(record)

    return transactions

This function extracts six field from each valid transaction line.
It will ignore lines which do not match the expected format. It will
return the transactions as a list of dictionaries.

Task 2: Recursive utility functions.

Part 1 of Task 2: Recursive directory search function.

In [27]:
def find_log_files(directory: str | Path) -> list[Path]:
    """Find all .log files inside the directory.
    
    Args: 
        directory: the path to the directory to search.
        
    Returns:
        A list of Path objects for all .log files that are found, 
        including files in nested subdirectories.
    """
    directory = Path(directory)
    log_files: list[Path] = []

    for item in directory.iterdir():
        if item.is_dir():
            log_files.extend(find_log_files(item))
        elif item.is_file() and item.suffix == ".log":
            log_files.append(item)

    return log_files

This function finds all the .log files which are inside the directory, examines 
the file paths and adds them in the subfolder to the list (.extend()) and adds an 
inidvidual .log file (.append()).

Part 2 of Task 2: Recursive function.

In [33]:
def find_active_registers(data: Any) -> set[str]:
    """Collect the active register IDs from the json file.
    
    Args:
        data: A decoded JSON value, with dictionaries, lists and individual values.

    Returns:
        A set of register IDs who has an True active value.
    """

    active_ids: set[str] = set()

    if isinstance(data, dict):
        register_id = data.get("id")

        if data.get("active") is True and isinstance(register_id, str):
            active_ids.add(register_id)

        for value in data.values():
            active_ids.update(find_active_registers(value))

    elif isinstance(data, list):
        for item in data:
            active_ids.update(find_active_registers(item))

    return active_ids

The function checks for active registers in dictionaries, search every item in the lists
and return an empty set.

In [36]:
with Path("data/store_config.json").open(encoding="utf-8") as file:
    store_config = json.load(file)

active_registers = find_active_registers(store_config)
print("Active register IDs:", sorted(active_registers))

Active register IDs: ['POS_01', 'POS_02', 'POS_03']


Loads the JSON into dictionaries and lists and finds the active register IDs.